# 02 - First look at the Oscars data

Goal: understand `the_oscar_award.csv` (read-only), find how Best Picture is labelled, no cleaning yet.

In [1]:
import duckdb

con = duckdb.connect()

In [3]:
# First 5 rows
con.sql("""
CREATE VIEW oscars AS
SELECT * FROM read_csv('../data/raw/oscar_archive_1927_2025/the_oscar_award.csv', ignore_errors=true)
""")
con.sql("SELECT * FROM oscars LIMIT 5")

┌───────────┬───────────────┬──────────┬──────────┬───────────────────────────┬─────────────────────┬────────────────────────┬─────────┐
│ year_film │ year_ceremony │ ceremony │ category │      canon_category       │        name         │          film          │ winner  │
│   int64   │     int64     │  int64   │ varchar  │          varchar          │       varchar       │        varchar         │ boolean │
├───────────┼───────────────┼──────────┼──────────┼───────────────────────────┼─────────────────────┼────────────────────────┼─────────┤
│      1927 │          1928 │        1 │ ACTOR    │ ACTOR IN A LEADING ROLE   │ Richard Barthelmess │ The Noose              │ false   │
│      1927 │          1928 │        1 │ ACTOR    │ ACTOR IN A LEADING ROLE   │ Richard Barthelmess │ The Patent Leather Kid │ false   │
│      1927 │          1928 │        1 │ ACTOR    │ ACTOR IN A LEADING ROLE   │ Emil Jannings       │ The Last Command       │ true    │
│      1927 │          1928 │        1 │ 

In [4]:
con.sql("SELECT COUNT(*) FROM oscars")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│        11235 │
└──────────────┘

In [5]:
# How many distinct film titles (all categories together)?
con.sql("SELECT COUNT(DISTINCT film) FROM oscars")

┌──────────────────────┐
│ count(DISTINCT film) │
│        int64         │
├──────────────────────┤
│                 5163 │
└──────────────────────┘

In [6]:
# Which category names exist? (we look for Best Picture)
con.sql("SELECT DISTINCT category FROM oscars ORDER BY category")

┌───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│                                                                 category                                                                  │
│                                                                  varchar                                                                  │
├───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ ACTOR                                                                                                                                     │
│ ACTOR IN A LEADING ROLE                                                                                                                   │
│ ACTOR IN A SUPPORTING ROLE                                                                                                                │
│ ACTR

In [7]:
# Same question for the canonical (harmonised) category names
con.sql("SELECT DISTINCT canon_category FROM oscars ORDER BY canon_category")

┌────────────────────────────────────────────┐
│               canon_category               │
│                  varchar                   │
├────────────────────────────────────────────┤
│ ACTOR IN A LEADING ROLE                    │
│ ACTOR IN A SUPPORTING ROLE                 │
│ ACTRESS IN A LEADING ROLE                  │
│ ACTRESS IN A SUPPORTING ROLE               │
│ ANIMATED FEATURE FILM                      │
│ ART DIRECTION                              │
│ ART DIRECTION (Black-and-White)            │
│ ART DIRECTION (Color)                      │
│ ASSISTANT DIRECTOR                         │
│ AWARD OF COMMENDATION                      │
│           ·                                │
│           ·                                │
│           ·                                │
│ SPECIAL ACHIEVEMENT AWARD (Sound Effects)  │
│ SPECIAL ACHIEVEMENT AWARD (Visual Effects) │
│ SPECIAL AWARD                              │
│ SPECIAL FOREIGN LANGUAGE FILM AWARD        │
│ UNIQUE AND 

In [8]:
con.sql("SELECT DISTINCT category, canon_category FROM oscars WHERE canon_category = 'ACTOR IN A LEADING ROLE'")

┌─────────────────────────┬─────────────────────────┐
│        category         │     canon_category      │
│         varchar         │         varchar         │
├─────────────────────────┼─────────────────────────┤
│ ACTOR IN A LEADING ROLE │ ACTOR IN A LEADING ROLE │
│ ACTOR                   │ ACTOR IN A LEADING ROLE │
└─────────────────────────┴─────────────────────────┘

In [9]:
con.sql("SELECT DISTINCT category, canon_category FROM oscars WHERE category LIKE '%PICTURE%' OR canon_category LIKE '%PICTURE%' ORDER BY canon_category, category")

┌─────────────────────────────┬─────────────────────────────┐
│          category           │       canon_category        │
│           varchar           │           varchar           │
├─────────────────────────────┼─────────────────────────────┤
│ BEST MOTION PICTURE         │ BEST PICTURE                │
│ BEST PICTURE                │ BEST PICTURE                │
│ OUTSTANDING MOTION PICTURE  │ BEST PICTURE                │
│ OUTSTANDING PICTURE         │ BEST PICTURE                │
│ OUTSTANDING PRODUCTION      │ BEST PICTURE                │
│ UNIQUE AND ARTISTIC PICTURE │ UNIQUE AND ARTISTIC PICTURE │
└─────────────────────────────┴─────────────────────────────┘

In [12]:
con.sql("SELECT year_ceremony, COUNT(*) AS nb_rows, COUNT(DISTINCT film) AS nb_films, SUM(CASE WHEN winner THEN 1 ELSE 0 END) AS nb_winners FROM oscars WHERE canon_category = 'BEST PICTURE' AND year_ceremony >= 1996 GROUP BY year_ceremony ORDER BY year_ceremony").show(max_rows=40)

┌───────────────┬─────────┬──────────┬────────────┐
│ year_ceremony │ nb_rows │ nb_films │ nb_winners │
│     int64     │  int64  │  int64   │   int128   │
├───────────────┼─────────┼──────────┼────────────┤
│          1996 │       5 │        5 │          1 │
│          1997 │       5 │        5 │          1 │
│          1998 │       5 │        5 │          1 │
│          1999 │       5 │        5 │          1 │
│          2000 │       5 │        5 │          1 │
│          2001 │       5 │        5 │          1 │
│          2002 │       5 │        5 │          1 │
│          2003 │       5 │        5 │          1 │
│          2004 │       5 │        5 │          1 │
│          2005 │       5 │        5 │          1 │
│          2006 │       5 │        5 │          1 │
│          2007 │       5 │        5 │          1 │
│          2008 │       5 │        5 │          1 │
│          2009 │       5 │        5 │          1 │
│          2010 │      10 │       10 │          1 │
│          2

In [13]:
con.sql("SELECT year_ceremony, year_film, film, winner FROM oscars WHERE canon_category = 'BEST PICTURE' AND year_ceremony BETWEEN 1996 AND 2015 ORDER BY year_ceremony, winner DESC, film").show(max_rows=200)

┌───────────────┬───────────┬───────────────────────────────────────────────────┬─────────┐
│ year_ceremony │ year_film │                       film                        │ winner  │
│     int64     │   int64   │                      varchar                      │ boolean │
├───────────────┼───────────┼───────────────────────────────────────────────────┼─────────┤
│          1996 │      1995 │ Braveheart                                        │ true    │
│          1996 │      1995 │ Apollo 13                                         │ false   │
│          1996 │      1995 │ Babe                                              │ false   │
│          1996 │      1995 │ Sense and Sensibility                             │ false   │
│          1996 │      1995 │ The Postman (Il Postino)                          │ false   │
│          1997 │      1996 │ The English Patient                               │ true    │
│          1997 │      1996 │ Fargo                                             